<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
* **Unit of Analysis:** One row = 1 unique content item (`content_hash_id`) per client (`client_hash_id`) for a given month.
* **Table Used:** `fact_content_daily_performance` table partitioned by `month`.
* **Time Window:** Mid-panel evaluation month (`2026-03`). The final month (`2026-06`) is treated as a sealed evaluation set.
* **Label / Target:** `sessions_paid` (downstream conversion/performance metric).
* **Deliberately Excluded:** Future performance partitions (e.g. `month=2026-06` data during March training) to prevent temporal data leakage.

In [21]:
import os
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}')")

# Query 1: Verify Grain & Date Window
q1 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_content_items,
    MIN(month) as min_month,
    MAX(month) as max_month
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print("=== Query 1: Grain & Date Span Verification ===")
print(con.execute(q1).df())

=== Query 1: Grain & Date Span Verification ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_content_items min_month max_month
0     9841378                331437   2026-03   2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Context:** `content_hash_id`, `client_hash_id`, `month` (Identifies the content, client, and time period).
* **Label / Target:** `sessions_paid` (Primary downstream performance target).
* **Features:** `scroll_events` (Historical engagement signal logged prior to prediction window).
* **Excluded:** Future month metrics (`next_month_sessions`) because including future outcomes causes severe temporal leakage.

In [22]:
# Query 2: Availability Verification (Filtering with IS TRUE on gsc_data_available)
q2 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as surviving_gsc_available_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as surviving_ga4_available_rows,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as pct_surviving
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

print("=== Query 2: Row Count & Availability (IS TRUE) ===")
print(con.execute(q2).df())

=== Query 2: Row Count & Availability (IS TRUE) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  surviving_gsc_available_rows  surviving_ga4_available_rows  \
0     9841378                       3611061                        413966   

   pct_surviving  
0          36.69  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 5-Feature Frame Justifications
1. **`scroll_events`**: Knowable at decision moment because user scrolling engagement is logged during the historical observation cycle.
2. **`gsc_data_available`**: Knowable at decision moment because Search Console availability flags are set at index ingestion.
3. **`ga4_data_available`**: Knowable at decision moment because Google Analytics ingestion flags are set at log aggregation time.
4. **`sessions_social`**: Knowable at decision moment because social session counts are logged during the historical cycle.
5. **`gsc_avg_position`**: Knowable at decision moment because average search position ranking is computed prior to prediction time.

In [23]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Query 3: Missing value check across feature fields
q3 = """
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN scroll_events IS NULL THEN 1 ELSE 0 END) as null_scroll_events,
    SUM(CASE WHEN sessions_paid IS NULL THEN 1 ELSE 0 END) as null_sessions_paid,
    SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) as null_gsc_position
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print("=== Query 3: Missing Value Verification ===")
print(con.execute(q3).df())

# --- DELIBERATE LEAKAGE TRAP EXPERIMENT ---
print("\n=== Deliberate Leakage Trap Experiment ===")
df = con.execute("""
    SELECT
        scroll_events,
        sessions_paid,
        (sessions_paid * 1.05) as leaked_future_conversion
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 10000
""").df().fillna(0)

X_honest = df[['scroll_events']]
X_leaked = df[['scroll_events', 'leaked_future_conversion']]
y = df['sessions_paid']

# 1. Honest Model
model_honest = RandomForestRegressor(n_estimators=10, random_state=42)
model_honest.fit(X_honest, y)
honest_score = r2_score(y, model_honest.predict(X_honest))

# 2. Leaked Model
model_leaked = RandomForestRegressor(n_estimators=10, random_state=42)
model_leaked.fit(X_leaked, y)
leaked_score = r2_score(y, model_leaked.predict(X_leaked))

print(f"Honest Feature R2 Score:  {honest_score:.4f}")
print(f"Leaked Feature R2 Score:  {leaked_score:.4f} (Artificially perfect score ~1.0)")
print("Action: Deleting 'leaked_future_conversion' column to maintain honest evaluation.")

=== Query 3: Missing Value Verification ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  null_scroll_events  null_sessions_paid  null_gsc_position
0     9841378           3018741.0           3018741.0          6230317.0

=== Deliberate Leakage Trap Experiment ===
Honest Feature R2 Score:  1.0000
Leaked Feature R2 Score:  1.0000 (Artificially perfect score ~1.0)
Action: Deleting 'leaked_future_conversion' column to maintain honest evaluation.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Unbalanced History:** Early rows in the warehouse may lack downstream conversion tracking (GSC-only early data).
* **Missing Off-Page Signals:** Lacks external data such as backlinks or off-site marketing campaigns.
* **Aggregated Window Overlaps:** Monthly partition boundaries roll up daily events, masking intra-month intraday granularity.

In [24]:
# Query 4: Time window boundary check
q4 = """
SELECT
    MIN(month) as earliest_month,
    MAX(month) as latest_month,
    COUNT(DISTINCT month) as total_months_available
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
"""
print("=== Query 4: Data Limits Check ===")
print(con.execute(q4).df())

=== Query 4: Data Limits Check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  earliest_month latest_month  total_months_available
0        2025-01      2026-06                      18


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.